# Нелінійні функції, softmax та attention

У попередній лабораторній ми побудували лінійний шар `linear(x, W, b)`:
$z = Wx + b$. Якщо скласти кілька таких шарів без функцій активації,
результат знову можна записати як один лінійний шар. Щоб мережа могла
моделювати складніші залежності, між шарами потрібна нелінійність.

Цього разу працюємо зі звичайними списками Python. Доповніть функції на
місці `...`, а потім запустіть `python3 attention.py`. Графіки до лабораторної
лежать у `figures/`.

## ReLU: пропускаємо додатні значення

$\operatorname{ReLU}(x)=\max(0,x)$. Для від'ємних чисел результат нульовий,
для додатних — саме число. Подивіться на `figures/activations.jpg`:
ReLU має злам у нулі. Що станеться з від'ємними компонентами вектора після
застосування ReLU?

![Графіки ReLU та GELU](figures/activations.jpg)

Нелінійність застосовується **після** лінійного шару, поелементно:
$h_i=\operatorname{ReLU}(z_i)$, де $z=Wx+b$.

In [ ]:
def relu(x):
    """Повертає ReLU одного числа."""
    ...


def test_relu():
    assert relu(-3) == 0
    assert relu(0) == 0
    assert relu(2.5) == 2.5


if __name__ == "__main__":
    test_relu()
    print("✓ relu")

## GELU: плавніша активація

GELU часто використовують у трансформерах. Її можна записати як
$\operatorname{GELU}(x)=x\Phi(x)$, де $\Phi$ — функція розподілу стандартної
нормальної величини. У Python її зручно обчислити через `math.erf`:

$$\operatorname{GELU}(x)=\frac{x}{2}
\left(1+\operatorname{erf}\left(\frac{x}{\sqrt{2}}\right)\right).$$

Порівняйте криві GELU та ReLU на графіку. GELU біля нуля плавна; для деяких
від'ємних $x$ вона теж повертає від'ємне, але мале число.

In [ ]:
def gelu(x):
    """Точна GELU через math.erf."""
    # Імпортуйте потрібні функції з math.
    ...


def test_gelu():
    from math import isclose

    assert gelu(0) == 0
    assert isclose(gelu(1), 0.8413447460685429, abs_tol=1e-12)
    assert isclose(gelu(-1), -0.15865525393145707, abs_tol=1e-12)
    assert isclose(gelu(3), 2.99595030590511, abs_tol=1e-12)


if __name__ == "__main__":
    test_gelu()
    print("✓ gelu")

## Поєднуємо активацію з лінійним шаром

Перенесіть або імпортуйте свою `linear` з першої лабораторної, якщо хочете
повторити обчислення. Тут уже готовий результат лінійного шару $z$.
Перетворіть кожен його елемент: спочатку через ReLU, потім через GELU.

```python
z = [-2.0, 0.0, 1.0]  # результат Wx + b
relu_vector(z)          # [0, 0, 1.0]
```

Питання: чому два лінійні шари поспіль можна згорнути в один, а два шари
з ReLU між ними — загалом ні?

In [ ]:
def relu_vector(values):
    """Поелементна ReLU для вектора."""
    ...


def gelu_vector(values):
    """Поелементна GELU для вектора."""
    ...


def test_activation_vectors():
    from math import isclose

    z = [-2.0, 0.0, 1.0]
    assert relu_vector(z) == [0, 0, 1.0]
    result = gelu_vector(z)
    assert len(result) == 3
    assert isclose(result[0], -0.04550026389635842, abs_tol=1e-12)
    assert result[1] == 0
    assert isclose(result[2], 0.8413447460685429, abs_tol=1e-12)
    assert z == [-2.0, 0.0, 1.0]


if __name__ == "__main__":
    test_activation_vectors()
    print("✓ activation vectors")

## Softmax: оцінки перетворюються на ваги

Для оцінок $s_1,\ldots,s_n$ softmax повертає
$p_i=e^{s_i}/\sum_j e^{s_j}$. Кожна вага додатна, а сума ваг дорівнює 1.
Softmax діє на **весь вектор**, на відміну від поелементних ReLU і GELU.
У багатокласовій класифікації модель спочатку видає довільні оцінки
(логіти) для кожного класу. Softmax перетворює їх на розподіл: числа від 0
до 1, сума яких **завжди дорівнює 1** для скінченних логітів. Тому їх часто
інтерпретують як імовірності класів. Проте сама нормалізація не гарантує,
що модель добре відкалібрована: вага 0.9 не обов'язково означає 90% успіху
на реальних даних. В attention ці самі числа — частки внеску значень, а не
ймовірності того, що певний ключ є «правильною відповіддю».

Наприклад, для оцінок класів `[0, ln(3)]` маємо ваги `[0.25, 0.75]`.
Другий клас отримує втричі більшу вагу, хоча різниця логітів — лише `ln(3)`.

На `figures/softmax.jpg` показано, як зміна однієї оцінки змінює всі ваги.
Спробуйте вручну обчислити softmax для `[0, 0]` та `[0, ln(3)]`.

![Ваги softmax для двох елементів](figures/softmax.jpg)

Але прямий виклик `exp(1000)` переповнює числа з плаваючою комою. Віднімання
одного й того самого числа від усіх оцінок не змінює softmax:

$$\frac{e^{s_i-c}}{\sum_j e^{s_j-c}}=
\frac{e^{s_i}}{\sum_j e^{s_j}}.$$

Оберіть $c=\max(s)$. Тоді найбільший показник експоненти дорівнює нулю,
а решта — від'ємні. Порожній вектор вважайте помилкою `ValueError`.
Припускаємо, що оцінки — скінченні числа.

In [ ]:
def softmax(scores):
    """Чисельно стійкий softmax для непорожнього вектора."""
    ...


def test_softmax():
    from math import isclose

    assert softmax([0, 0]) == [0.5, 0.5]
    assert softmax([0]) == [1.0]
    result = softmax([1, 2, 3])
    shifted = softmax([1001, 1002, 1003])
    assert all(isclose(a, b, abs_tol=1e-12) for a, b in zip(result, shifted))
    assert isclose(sum(result), 1.0, abs_tol=1e-12)
    assert softmax([1000, 1000]) == [0.5, 0.5]
    assert softmax([-1000, -1000]) == [0.5, 0.5]

    try:
        softmax([])
    except ValueError:
        pass
    else:
        raise AssertionError("Порожній вектор має давати ValueError")


if __name__ == "__main__":
    test_softmax()
    print("✓ softmax")

## Від лінійного шару до порівняння векторів

У першій лабораторній ми обчислювали `linear(x, W, b) = Wx + b`.
Для attention модель застосовує **різні навчені лінійні проєкції** до
вхідних векторів: з поточного вектора $x$ робить запит $q=W_Qx+b_Q$,
а з кожного доступного вектора $x_i$ — ключ $k_i=W_Kx_i+b_K$ і значення
$v_i=W_Vx_i+b_V$. Матриці $W_Q$, $W_K$ і $W_V$ — параметри моделі.
Запит і ключ мають однакову довжину $d_k$, щоб їх можна було порівняти.
Довжина значення може бути іншою.

Перший крок — `dot(q, k_i)`. Ось обчислення з функціями першої лабораторної
(одиничні проєкції й нульові зсуви для простоти):

```python
W_q = [[1, 0], [0, 1]]
W_k = [[1, 0], [0, 1]]
q = linear([1, 0], W_q, [0, 0])    # [1, 0]
k_1 = linear([1, 0], W_k, [0, 0])  # [1, 0]
k_2 = linear([0, 1], W_k, [0, 0])  # [0, 1]
dot(q, k_1), dot(q, k_2)           # (1, 0)
```

Перший ключ краще узгоджується із запитом.
Додатний добуток означає співнапрямленість, від'ємний — протилежність,
нульовий — ортогональність. Але це не відстань: довший вектор може дати
більший добуток лише через свій масштаб. Навчені проєкції дають моделі
змогу визначити, які ознаки порівнювати, а навчання підлаштовує і напрямки,
і масштаби.

Чому потрібні дві проєкції? Одна й та сама позиція може *шукати* одні
ознаки через свій запит і *пропонувати* інші через свій ключ. Наприклад,
у реченні поточне слово може шукати попередній іменник, а ключі попередніх
слів показують, яке з них відповідає цьому запиту.

## Attention для одного запиту

Тепер маємо один запит, кілька пар «ключ — значення» і три кроки.

1. **Порівняти:** `dot(q, k_i)` дає оцінку для кожного ключа. Ділимо її
   на $\sqrt{d_k}$, щоб за великої кількості координат оцінки не зростали
   надто сильно.
2. **Розподілити увагу:** softmax перетворює всі оцінки разом на невід'ємні
   ваги із сумою 1. Ключі з більшими оцінками отримують більший внесок,
   але кілька ключів можуть впливати одночасно. Це плавний, придатний до
   навчання спосіб розподілити «скільки дивитися» на кожну позицію.
3. **Зібрати інформацію:** кожне значення $v_i$ множимо на його вагу
   $\alpha_i$ і додаємо. Отримуємо новий вектор для поточної позиції.
   Саме значення переносить інформацію; ключ лише допомагає вирішити,
   яку частку цього значення використати.

Отже, це два різні множення. $q\cdot k_i$ відповідає на питання
«наскільки цей ключ підходить запиту?». Фінальне $\alpha V$ відповідає
«яку суміш значень взяти?». Для кожної координати результату це теж
скалярний добуток вектора ваг із відповідним стовпцем $V$, але вже не
порівняння подібності. Так можна розуміти attention як **перезважування
інформації**, яку отримує кожна позиція мережі.

$$s_i=\frac{q\cdot k_i}{\sqrt{d_k}},\qquad
\alpha=\operatorname{softmax}(s),\qquad
o=\sum_i\alpha_i v_i.$$

Розгляньмо `query = [1, 0]` і три ключі `[1, 0]`, `[0, 1]`, `[-1, 0]`.
Їхні скалярні добутки із запитом: `[1, 0, -1]`. Після ділення на
$\sqrt{2}$ та softmax ваги приблизно `[0.576, 0.284, 0.140]`.
Перший ключ узгоджується із запитом найбільше, але інші не зникають:
softmax дозволяє змішувати інформацію з кількох позицій. Якщо значення —
`[10, 0]`, `[0, 20]`, `[-10, 0]`, вихід приблизно `[4.36, 5.68]`.
Простежте кожен крок обчислення й перевірте зважену суму вручну.

Коли компоненти запиту й ключів мають приблизно одиничний масштаб,
дисперсія їхнього скалярного добутку зростає разом із $d_k$.
Великі за модулем оцінки роблять softmax надто різким; масштабування
стримує цей ефект. Для кількох запитів одразу отримуємо
$\operatorname{softmax}(QK^T/\sqrt{d_k})V$ (softmax у кожному рядку).
Цю форму scaled dot-product attention описано в статті
[Attention Is All You Need](https://arxiv.org/abs/1706.03762), розділ 3.2.1.

Реалізуйте `dot` тут або імпортуйте свою функцію з першої лабораторної.
Вимагайте однакову додатну довжину запиту та всіх ключів; ключів має бути
стільки ж, скільки значень. Усі значення повинні мати однакову додатну
довжину. За некоректних форм піднімайте `ValueError`.

In [ ]:
def dot(a, b):
    """Скалярний добуток; вектори різної довжини заборонені."""
    ...


def attention(query, keys, values):
    """Повертає (результат, ваги) для одного запиту."""
    # 1. Перевірте форми.
    # 2. Обчисліть оцінки через dot та поділіть на sqrt(len(query)).
    # 3. Отримайте ваги через softmax.
    # 4. Знайдіть зважену суму значень по кожній координаті.
    ...


def test_attention():
    from math import isclose

    keys = [[1, 0], [0, 1]]
    values = [[10, 0], [0, 20]]
    output, weights = attention([0, 0], keys, values)
    assert weights == [0.5, 0.5]
    assert output == [5.0, 10.0]

    output, weights = attention([1, 0], keys, values)
    assert weights[0] > weights[1]
    assert isclose(sum(weights), 1.0, abs_tol=1e-12)
    assert isclose(output[0], 10 * weights[0], abs_tol=1e-12)
    assert isclose(output[1], 20 * weights[1], abs_tol=1e-12)

    for bad_query, bad_keys, bad_values in [
        ([], keys, values),
        ([1, 0], [], []),
        ([1, 0], keys, [[1, 2]]),
        ([1, 0], [[1], [0, 1]], values),
        ([1, 0], keys, [[1], [2, 3]]),
    ]:
        try:
            attention(bad_query, bad_keys, bad_values)
        except ValueError:
            pass
        else:
            raise AssertionError("Некоректні форми мають давати ValueError")


if __name__ == "__main__":
    test_attention()
    print("✓ attention")

## Multi-head attention: кілька поглядів на ті самі токени

Попередня функція обчислює attention для **одного** запиту. У self-attention
кожен токен послідовності створює свій запит і порівнюється з ключами
токенів цієї ж послідовності. Одна head має власні матриці $W_Q$, $W_K$,
$W_V$. Різні heads мають різні матриці, тому можуть навчитися звертати
увагу на різні ознаки й позиції. Це можливість, а не гарантія, що кожна
head матиме просте людське тлумачення.

Для послідовності `xs` довжини $T$ із векторами довжини $d_{model}$ одна
head створює матриці $Q,K$ форми $(T,d_k)$ та $V$ форми $(T,d_v)$.
Обчисліть `attention(q, keys, values)` для **кожного** рядка $q$ з $Q$.
Отримаєте $T$ вихідних векторів довжини $d_v$ і матрицю ваг $(T,T)$:
рядок — позиція запиту, стовпець — позиція ключа. Softmax обчислюється
окремо в кожному рядку, тому сума кожного рядка дорівнює 1.

Якщо heads $H$, для кожного токена з'єднайте їхні виходи в один довгий
вектор. Потім застосуйте вихідну матрицю $W_O$. У цій вправі зсуви
пропускаємо, щоб зосередитися на потоці даних:

$$\operatorname{MultiHead}(X)=
\operatorname{Concat}(\operatorname{head}_1(X),\ldots,
\operatorname{head}_H(X))W_O^T.$$

Кожна head задається трійкою `(W_q, W_k, W_v)`. Як і в першій
лабораторній, **рядки** матриці ваг — вихідні ознаки, а стовпці — вхідні.
`output_weight` має форму `(out_dim, sum_d_v)`. Поверніть пару:
`(outputs, weights_by_head)`, де `outputs` має форму `(T, out_dim)`,
а `weights_by_head` — `(H, T, T)`. Для простоти всі токени мають бути
векторами однакової додатної довжини; heads і послідовність — непорожні.
Некоректні форми мають давати `ValueError`.

Приклад із двома heads: перша дивиться лише на першу координату, друга —
лише на другу. Для `xs = [[1, 0], [0, 1]]` і одиничної вихідної матриці
перша позиція дає приблизно `[0.731, 0.5]`, друга — `[0.5, 0.731]`.
Спробуйте пояснити ці числа через ваги кожної head.

In [ ]:
def multi_head_attention(xs, heads, output_weight):
    """Повертає виходи токенів і матрицю ваг для кожної head."""
    # 1. Перевірте форми входу та кожної матриці проєкції.
    # 2. Для кожної head обчисліть Q, K, V через dot рядків ваг із x.
    # 3. Застосуйте attention до кожного запиту цієї head.
    # 4. З'єднайте виходи heads для кожного токена.
    # 5. Застосуйте output_weight до кожного з'єднаного вектора.
    ...


def test_multi_head_attention():
    from math import exp, isclose

    xs = [[1, 0], [0, 1]]
    heads = [
        ([[1, 0]], [[1, 0]], [[1, 0]]),
        ([[0, 1]], [[0, 1]], [[0, 1]]),
    ]
    output_weight = [[1, 0], [0, 1]]
    outputs, weights = multi_head_attention(xs, heads, output_weight)

    strong = 1 / (1 + exp(-1))
    assert len(outputs) == 2 and all(len(row) == 2 for row in outputs)
    assert len(weights) == 2
    assert all(len(matrix) == 2 for matrix in weights)
    assert all(len(row) == 2 for matrix in weights for row in matrix)
    for got, expected in zip(outputs, [[strong, 0.5], [0.5, strong]]):
        assert all(isclose(a, b, abs_tol=1e-12) for a, b in zip(got, expected))
    assert all(isclose(sum(row), 1.0, abs_tol=1e-12)
               for matrix in weights for row in matrix)
    assert isclose(weights[0][0][0], strong, abs_tol=1e-12)
    assert weights[1][0] == [0.5, 0.5]
    assert isclose(weights[1][1][1], strong, abs_tol=1e-12)
    assert xs == [[1, 0], [0, 1]]

    for bad_xs, bad_heads, bad_output in [
        ([], heads, output_weight),
        ([[1, 0], [1]], heads, output_weight),
        (xs, [], output_weight),
        (xs, [([[1, 0]], [[1]], [[1, 0]])], [[1]]),
        (xs, [([[1, 0]], [[1, 0]], [[1, 0]])], [[1, 0]]),
    ]:
        try:
            multi_head_attention(bad_xs, bad_heads, bad_output)
        except ValueError:
            pass
        else:
            raise AssertionError("Некоректні форми мають давати ValueError")


if __name__ == "__main__":
    test_multi_head_attention()
    print("✓ multi_head_attention")

## Як змінюються ваги з появою нових токенів?

«З часом» може означати дві речі. Під час генерації тексту модель отримує
наступний токен і створює **новий запит**. У causal self-attention цей запит
бачить лише поточний та попередні токени, тому довжина його рядка ваг
зростає. Ваги для нового запиту перераховуються між усіма доступними
ключами. Уже обчислений рядок для старого запиту тут не «переписується».
Вправа `multi_head_attention` вище обчислює повний self-attention без
causal mask; нижче ми просто передаємо `attention` доступний префікс.

Уявімо переклад **«The cat sat on the mat.» → «Кіт сидів на килимку.»**.
Encoder обробляє **все** англійське речення й готує ключі та значення
джерела. Decoder починає з `BOS` (початок речення) і генерує українські
токени один за одним до `EOS` (кінець речення). На кожному кроці він
використовує два різні джерела інформації:

- **Causal self-attention:** запит із поточного українського префікса
  порівнюється тільки з його власними попередніми й поточним токенами.
  Майбутній переклад прихований маскою.
- **Cross-attention:** запит decoder порівнюється з ключами **англійського**
  речення; зважена сума значень encoder передає інформацію про джерело.
  Тут усі англійські позиції вже доступні, тому causal mask не потрібна.

Потім модель обчислює оцінки для токенів українського словника; softmax
дає розподіл для **наступного слова**. Це ще один softmax, окремий від
softmax, який створює ваги attention. Послідовність кроків така:

| Поточний український префікс | Можлива підказка з англійського джерела | Наступний токен |
| --- | --- | --- |
| `BOS` | `cat` | `Кіт` |
| `BOS Кіт` | `sat` | `сидів` |
| `BOS Кіт сидів` | `on` | `на` |
| `BOS Кіт сидів на` | `mat` | `килимку` |
| `BOS Кіт сидів на килимку` | `.` | `.` |
| `BOS Кіт сидів на килимку .` | кінець джерела | `EOS` |

Середній стовпець — **педагогічні підказки**, а не виміряні ваги
перекладача: реальна модель може враховувати кілька англійських слів
одночасно, а порядок і форми українських слів залежать також від префікса.
Саме так переклад пов'язує дві послідовності. Виклик
`attention(q_target, keys_source, values_source)` описує одну cross-attention
head; попередня вправа використовувала запити, ключі та значення з **тієї
самої** послідовності.

Для **числового прикладу першого кроку** залишимо лише три англійські
слова `The cat sat`. Припустімо, що запит від `BOS` дорівнює `[1, 0]`,
а encoder дав такі ключі й значення:

```python
keys_source = [[0, 1], [1, 0], [0, 1]]   # The, cat, sat
values_source = [[0, 0], [1, 0], [0, 1]]
context, source_weights = attention([1, 0], keys_source, values_source)
# source_weights ≈ [0.248, 0.503, 0.248]
# context ≈ [0.503, 0.248]
```

Запит найбільше зважив `cat`. Якщо в цій **іграшковій** моделі дві
координати `context` вважати оцінками слів `[Кіт, сидів]`, softmax дає
приблизно `[0.563, 0.437]`, тож наступним токеном стає `Кіт`.
У справжньому перекладачі контекст поєднується з іншими сигналами decoder,
а навчена вихідна проєкція створює логіти для всього словника. Ми задали
вектори вручну, щоб показати шлях інформації, а не навчили перекладач.

Тепер розгляньмо детальніше лише causal self-attention decoder. У таблиці
нижче кожен рядок відповідає **останньому токену поточного префікса** і
допомагає передбачити наступний токен із таблиці вище.

Візьмемо прості двовимірні вектори й одиничні проєкції. Це **іграшкові
числа для демонстрації маски**, а не ваги навченої моделі перекладу.
Рядок — токен, який зараз створює запит; стовпець — доступний ключ.
`—` означає майбутню позицію, яка не бере участі в softmax.

| Запит ↓ / Ключ → | BOS | Кіт | сидів | на | килимку | . |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| BOS       | 1.000 | — | — | — | — | — |
| Кіт       | 0.330 | 0.670 | — | — | — | — |
| сидів     | 0.248 | 0.248 | 0.503 | — | — | — |
| на        | 0.190 | 0.270 | 0.270 | 0.270 | — | — |
| килимку   | 0.117 | 0.237 | 0.166 | 0.198 | 0.282 | — |
| .         | 0.184 | 0.129 | 0.184 | 0.154 | 0.129 | 0.220 |

Кожен рядок сумується до 1 (з похибкою округлення). Наприклад, запит
«сидів» може використати BOS, «Кіт» і себе, але ще не «на» чи «килимку»;
цей рядок допомагає передбачити «на».
Поява нового токена додає **новий рядок** і стовпець у загальну матрицю;
попередні рядки залишаються такими самими. Виконайте приклад нижче після
реалізації `attention` і порівняйте числа. У multi-head attention кожна
head матиме свою матрицю ваг із такою ж трикутною маскою.

Інший сенс часу — **навчання**. Після оновлення $W_Q$ і $W_K$ ті самі
вхідні токени можуть дати інші оцінки та ваги. На це впливає навчання
параметрів, а не саме додавання нового токена. Ваги корисно досліджувати,
але самі по собі вони не доводять, чому модель ухвалила певне рішення.

In [ ]:
def show_causal_weights():
    """Друкує трикутну матрицю ваг іграшкового прикладу перекладу."""
    words = ["BOS", "Кіт", "сидів", "на", "килимку", "."]
    tokens = [[0, 0], [1, 0], [0, 1], [0.5, 0.5],
              [1, 0.5], [-0.5, 0]]
    print("запит/ключ", *words, sep="\t")
    for step, word in enumerate(words):
        _, weights = attention(tokens[step], tokens[:step + 1], tokens[:step + 1])
        cells = [f"{weight:.3f}" for weight in weights]
        cells += ["—"] * (len(tokens) - len(cells))
        print(word, *cells, sep="\t")


if __name__ == "__main__":
    show_causal_weights()

## Питання для обговорення

1. Чому softmax `[1000, 1001]` можна обчислити після віднімання 1001?
2. Що станеться з вагами attention, якщо помножити всі оцінки на велике число?
3. Чому `values` можуть мати іншу розмірність, ніж `keys`?
4. Які матричні операції з першої лабораторної допоможуть обчислити attention
   одразу для кількох запитів? Запишіть форми матриць $Q$, $K$, $V$,
   $QK^T$ та результату.
5. Чому дві heads можуть дати різні ваги для тих самих токенів?
6. У таблиці вище, чи змінюється рядок «Кіт», коли з'являється «сидів»?
   Що має статися, щоб ті самі токени під час навчання отримали інші ваги?
7. Чому англійські слова з прикладу перекладу не стоять у стовпцях
   causal self-attention? Який механізм може звертатися до них?